In [ ]:
!pip install --disable-pip-version-check -q optuna

In [1]:
import json
import sys
import time
from datetime import datetime

import boto3
import numpy as np
import optuna
import pandas as pd
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import ModelMetrics
from sklearn.metrics import fbeta_score

sys.path.append('../config')
import config

C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\dcode\AppData\Local\sagemaker\sagemaker\config.yaml


In [2]:
# Set up AWS session
session = boto3.session.Session()
region = session.region_name
sagemaker_session = sagemaker.Session()
s3_client = boto3.client('s3')
role = "arn:aws:iam::637423636147:role/LabRole"
# role = get_execution_role()

In [3]:
# Set up S3 vars
bucket = config.S3_BUCKET
s3_prefix = 'final_project/feature_engineer'
model_prefix = 'final_project/models/'
output_prefix = 'final_project/output/'

train_path = f's3://{bucket}/{s3_prefix}/Xy_train.csv'
val_path = f's3://{bucket}/{s3_prefix}/Xy_val.csv'

In [4]:
# Load data from S3
def load_data_from_s3(s3_path):
    bucket_name = bucket
    key = s3_path.replace(f's3://{bucket_name}/', '')
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    return pd.read_csv(obj['Body'])


print(f"Attempting to load training data from {train_path}.")
train_df = load_data_from_s3(train_path)
print(f"Attempting to load validation data from {val_path}.")
val_df = load_data_from_s3(val_path)

print(f"Training data shape: {train_df.shape}")
print(f"Validation data shape: {val_df.shape}")

X_train = train_df.drop('label', axis=1)
y_train = train_df['label']
X_val = val_df.drop('label', axis=1)
y_val = val_df['label']

Attempting to load training data from s3://sagemaker-us-east-1-637423636147/final_project/feature_engineer/Xy_train.csv.
Attempting to load validation data from s3://sagemaker-us-east-1-637423636147/final_project/feature_engineer/Xy_val.csv.
Training data shape: (88947, 13)
Validation data shape: (21905, 13)


In [10]:
# Define Optuna objective function for hyperparameter tuning
def objective(trial):
    hyperparameters = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'alpha': trial.suggest_float('alpha', 0, 10),
        'lambda': trial.suggest_float('lambda', 0, 10),
        'num_round': 100,
        'objective': 'binary:logistic',
        'eval_metric': 'f1',
        'csv_label_column': str(train_df.shape[1] - 1)
    }

    job_name = f"xgboost-tuning-{int(time.time())}"
    xgb = sagemaker.estimator.Estimator(
        image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.5-1"),
        role=role,
        instance_count=1,
        instance_type='ml.m5.xlarge',
        output_path=f's3://{bucket}/{output_prefix}',
        sagemaker_session=sagemaker_session,
        hyperparameters=hyperparameters
    )

    train_input = TrainingInput(
        s3_data=train_path,
        content_type='text/csv'
    )

    validation_input = TrainingInput(
        s3_data=val_path,
        content_type='text/csv'
    )

    xgb.fit(
        {'train': train_input, 'validation': validation_input},
        job_name=job_name,
        wait=True,
        logs=False
    )

    model = sagemaker.model.Model(
        image_uri=xgb.image_uri,
        model_data=xgb.model_data,
        role=role,
        sagemaker_session=sagemaker_session
    )

    transformer = model.transformer(
        instance_count=1,
        instance_type='ml.m5.xlarge',
        output_path=f's3://{bucket}/{output_prefix}/predictions'
    )

    transformer.transform(val_path, content_type='text/csv', split_type='Line')
    transformer.wait()

    prediction_key = f"{output_prefix}/predictions/{val_path.split('/')[-1]}.out"
    obj = s3_client.get_object(Bucket=bucket, Key=prediction_key)
    predictions = np.loadtxt(obj['Body'])

    binary_predictions = (predictions > 0.5).astype(int)
    f2_score = fbeta_score(y_val.values, binary_predictions, beta=2)
    return f2_score

In [11]:
# Run Optuna hyperparameter tuning
print("Starting hyperparameter tuning with Optuna...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2025-06-07 11:57:17,287] A new study created in memory with name: no-name-a665d05e-8c68-430a-aea4-46be88059f3d
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: xgboost-tuning-1749311837


Starting hyperparameter tuning with Optuna...

2025-06-07 15:57:19 Starting - Starting the training job..
2025-06-07 15:57:33 Starting - Preparing the instances for training..
2025-06-07 15:57:52 Downloading - Downloading input data....
2025-06-07 15:58:17 Downloading - Downloading the training image........
2025-06-07 15:59:03 Training - Training image download completed. Training in progress...
2025-06-07 15:59:19 Uploading - Uploading generated training model
2025-06-07 15:59:22 Failed - Training job failed


[W 2025-06-07 11:59:26,510] Trial 0 failed with parameters: {'max_depth': 6, 'eta': 0.03164040061968793, 'gamma': 4.466401322074366, 'min_child_weight': 1, 'subsample': 0.6737040779244448, 'colsample_bytree': 0.9114924997940821, 'alpha': 8.38525902467792, 'lambda': 5.395972766848027} because of the following error: UnexpectedStatusException('Error for Training job xgboost-tuning-1749311837: Failed. Reason: AlgorithmError: framework error: \nTraceback (most recent call last):\n  File "/miniconda3/lib/python3.8/site-packages/sagemaker_algorithm_toolkit/hyperparameter_validation.py", line 276, in validate\n    hyperparameter_obj = self.hyperparameters[hp]\nKeyError: \'csv_label_column\'\n\nDuring handling of the above exception, another exception occurred:\n\nTraceback (most recent call last):\n  File "/miniconda3/lib/python3.8/site-packages/sagemaker_containers/_trainer.py", line 84, in train\n    entrypoint()\n  File "/miniconda3/lib/python3.8/site-packages/sagemaker_xgboost_container/t

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 # Run Optuna hyperparameter tuning                                                           │
│   2 print("Starting hyperparameter tuning with Optuna...")                                       │
│   3 study = optuna.create_study(direction='maximize')                                            │
│ ❱ 4 study.optimize(objective, n_trials=10)                                                       │
│   5                                                                                              │
│                                                                                                  │
│ C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\optuna\study\study.py:475 in optimize │
│                                                                                                  │
│    472 │   │   │   RuntimeError:                                                                 │
│    473 │   │   │   │   If nested invocation of this method occurs.                               │
│    474 │   │   """                                                                               │
│ ❱  475 │   │   _optimize(                                                                        │
│    476 │   │   │   study=self,                                                                   │
│    477 │   │   │   func=func,                                                                    │
│    478 │   │   │   n_trials=n_trials,                                                            │
│                                                                                                  │
│ C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\optuna\study\_optimize.py:63 in       │
│ _optimize                                                                                        │
│                                                                                                  │
│    60 │                                                                                          │
│    61 │   try:                                                                                   │
│    62 │   │   if n_jobs == 1:                                                                    │
│ ❱  63 │   │   │   _optimize_sequential(                                                          │
│    64 │   │   │   │   study,                                                                     │
│    65 │   │   │   │   func,                                                                      │
│    66 │   │   │   │   n_trials,                                                                  │
│                                                                                                  │
│ C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\optuna\study\_optimize.py:160 in      │
│ _optimize_sequential                                                                             │
│                                                                                                  │
│   157 │   │   │   │   break                                                                      │
│   158 │   │                                                                                      │
│   159 │   │   try:                                                                               │
│ ❱ 160 │   │   │   frozen_trial = _run_trial(study, func, catch)                                  │
│   161 │   │   finally:                                                                           │
│   162 │   │   │   # The following line mitigates memory problems that can be occurred in some    │
│   163 │   │   │   # environments (e.g., services that use c

In [ ]:
# Print best hyperparameters and score
print("Best hyperparameters:", study.best_params)
print("Best validation F2 score:", study.best_value)

In [8]:
# Train final model with best hyperparameters
best_params = study.best_params
best_params.update({
    'num_round': 100,
    'objective': 'binary:logistic',
    'eval_metric': 'f1',
    'csv_label_column': str(train_df.shape[1] - 1)
})

final_job_name = f"xgboost-v1-{int(time.time())}"
final_xgb = sagemaker.estimator.Estimator(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region, "1.5-1"),
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/{model_prefix}',
    sagemaker_session=sagemaker_session,
    hyperparameters=best_params
)

train_input = TrainingInput(
    s3_data=train_path,
    content_type='text/csv'
)

validation_input = TrainingInput(
    s3_data=val_path,
    content_type='text/csv'
)

print("Training final model with best hyperparameters...")
final_xgb.fit(
    {'train': train_input, 'validation': validation_input},
    job_name=final_job_name,
    wait=True
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: xgboost-v1-1749310770


Training final model with best hyperparameters...
2025-06-07 15:39:34 Starting - Starting the training job...
2025-06-07 15:40:07 Downloading - Downloading input data...
2025-06-07 15:40:32 Downloading - Downloading the training image......
2025-06-07 15:41:42 Training - Training image download completed. Training in progress.
2025-06-07 15:41:42 Uploading - Uploading generated training model
2025-06-07 15:41:42 Failed - Training job failed
/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2025-06-07 15:41:25.516 ip-10-0-191-98.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2025-06-07 15:41:25.540 ip-10-0-191-98.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2025-06-07:15:41:25:INFO] Imported framework sagemaker_xgboost_cont

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:30                                                                                   │
│                                                                                                  │
│   27 )                                                                                           │
│   28                                                                                             │
│   29 print("Training final model with best hyperparameters...")                                  │
│ ❱ 30 final_xgb.fit(                                                                              │
│   31 │   {'train': train_input, 'validation': validation_input},                                 │
│   32 │   job_name=final_job_name,                                                                │
│   33 │   wait=True                                                                               │
│                                                                                                  │
│ C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\sagemaker\telemetry\telemetry_logging │
│ .py:167 in wrapper                                                                               │
│                                                                                                  │
│   164 │   │   │   │   │   caught_ex = e                                                          │
│   165 │   │   │   │   finally:                                                                   │
│   166 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 167 │   │   │   │   │   │   raise caught_ex                                                    │
│   168 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   169 │   │   │   else:                                                                          │
│   170 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\sagemaker\telemetry\telemetry_logging │
│ .py:138 in wrapper                                                                               │
│                                                                                                  │
│   135 │   │   │   │   start_timer = perf_counter()                                               │
│   136 │   │   │   │   try:                                                                       │
│   137 │   │   │   │   │   # Call the original function                                           │
│ ❱ 138 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   139 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   140 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   141 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ C:\Repos\AAI540_FinalProject_Team5\.venv\Lib\site-packages\sagemaker\workflow\pipeline_context.p │
│ y:346 in wrapper                                                                                 │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)             

In [ ]:
# Calculate F2 score on validation data for final model
transformer = final_xgb.transformer(
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/{output_prefix}/final_predictions'
)

transformer.transform(val_path, content_type='text/csv', split_type='Line')
transformer.wait()

prediction_key = f"{output_prefix}/final_predictions/{val_path.split('/')[-1]}.out"
obj = s3_client.get_object(Bucket=bucket, Key=prediction_key)
predictions = np.loadtxt(obj['Body'])

binary_predictions = (predictions > 0.5).astype(int)
final_f2_score = fbeta_score(y_val.values, binary_predictions, beta=2)

In [ ]:
# Register model in Model Registry
model_info = {
    'model_data': final_xgb.model_data,
    'training_job_name': final_xgb.latest_training_job.name,
    'hyperparameters': best_params,
    'validation_f2_score': float(final_f2_score),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

model_info_key = f'{model_prefix}model_info.json'
s3_client.put_object(
    Body=json.dumps(model_info, indent=2),
    Bucket=bucket,
    Key=model_info_key
)

print("Registering model in Model Registry...")
model_package_group_name = f"ddos-prediction-model-group-{int(time.time())}"
model_metrics = ModelMetrics(
    model_statistics=None,
    model_data_statistics=None,
    bias=None,
    explainability=None,
)

In [ ]:
# Create model package group
sm_client = boto3.client('sagemaker')
try:
    sm_client.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription="Credit Default Prediction Models"
    )
except sm_client.exceptions.ResourceInUse:
    print(f"Model package group {model_package_group_name} already exists")

# Create a model from the training job
model = sagemaker.model.Model(
    image_uri=final_xgb.image_uri,
    model_data=final_xgb.model_data,
    role=role,
    sagemaker_session=sagemaker_session
)

model_package_arn = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status="Approved",
    model_metrics=model_metrics
)

print(f"Model training completed. Model info saved to s3://{bucket}/{model_info_key}")
print(f"Model artifact path: {final_xgb.model_data}")
print(f"Model package ARN: {model_package_arn}")